**Unit 2 Assignment: Building a Mixture of Experts (MoE) Router**
Topic: Advanced Architecture using Groq API
Estimated Time: 45-60 Minutes
Tools: Python, Groq API, Dotenv

🎯 Objective
Your task is to build a "Smart Customer Support Router" using a Mixture of Experts (MoE) architecture.

In a real-world company, you don't want a "Generalist" AI handling everything. You want:

- A Technical Expert for bug reports.
- A Billing Expert for refund requests.
- A Sales Expert for new inquiries.
You will build a Router that takes a user query, decides which expert is best suited for it, and then forwards the query to that specific expert configuration.

In [12]:
# pip install groq python-dotenv

import os
import getpass
from groq import Groq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

- API key is securely loaded using getpass to prevent credential leakage.

- Groq client is initialized once for reuse across the application.

In [13]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": (
            "You are a senior software engineer. "
            "Be precise, technical, and code-focused. "
            "Explain bugs clearly and suggest fixes."
        ),
        "temperature": 0.7
    },
    "billing": {
        "system_prompt": (
            "You are a customer billing support agent. "
            "Be empathetic, polite, and policy-driven. "
            "Explain charges, refunds, and subscriptions clearly."
        ),
        "temperature": 0.7
    },
    "general": {
        "system_prompt": (
            "You are a helpful general assistant. "
            "Answer clearly and concisely."
        ),
        "temperature": 0.7
    }
}


In [17]:
def route_prompt(user_input: str) -> str:
    prompt = f"""
    Classify the following user query into one category:
    [technical, billing, general]

    Query: "{user_input}"

    Return ONLY the category name.
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip().lower()


In [18]:
def process_request(user_input: str) -> str:
    category = route_prompt(user_input)

    expert = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": expert["system_prompt"]},
            {"role": "user", "content": user_input}
        ],
        temperature=expert["temperature"]
    )


    return response.choices[0].message.content


In [19]:
queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "Tell me a fun fact about space."
]

for q in queries:
    print("\nUser:", q)
    print("Detected Category:", route_prompt(q))
    print("Response:", process_request(q))



User: My python script is throwing an IndexError on line 5.
Detected Category: technical
Response: To help you debug the issue, I'll need more information about your script, including:

1. The code snippet that's causing the error.
2. The line number (which you've mentioned is 5).
3. The exact error message.

However, I can explain what an `IndexError` typically means in Python:

An `IndexError` occurs when you try to access an element in a sequence (like a list, tuple, or string) using an index that's out of range. This means you're trying to access an element that doesn't exist.

Here's a general example of how this might look:

```python
my_list = [1, 2, 3]
print(my_list[3])  # IndexError: list index out of range
```

To fix the issue, you'll need to determine why the index is out of range. This could be due to the index being larger than the length of the sequence or due to an incorrect index calculation.

If you provide the actual code and the error message, I can give you a more

In [20]:
def get_bitcoin_price():
    return "Current Bitcoin price is approximately $42,000 (mock data)."

def process_request_with_tools(user_input: str) -> str:
    if "bitcoin" in user_input.lower() or "price" in user_input.lower():
        return get_bitcoin_price()

    return process_request(user_input)


In [21]:
print(process_request_with_tools("What is the current price of Bitcoin?"))

Current Bitcoin price is approximately $42,000 (mock data).
